<a href="https://colab.research.google.com/github/va4756/bigdata_raejung/blob/main/LLM_product_chap2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 2-6 RNN과 LSTM

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install gensim
!python -m spacy download en_core_web_lg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 39.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
import nltk
import spacy

In [4]:
# nltk.download('stopwords')
# tokenizer = nltk.tokenize.RegexpTokenizer(r"\w+'?\w+|\w+'")
# tokenizer.tokenize("I'm studying NLP and it's interesting.")
# stop_words = nltk.corpus.stopwords.words("english")
# print(stop_words[:20])
# print(len(stop_words))

In [5]:
try:
    tokenizer = nltk.tokenize.RegexpTokenizer(r"\w+'?\w+|\w+'")
    tokenizer.tokenize("I'm studying NLP and it's interesting.")
    stop_words = nltk.corpus.stopwords.words("english")
    nlp = spacy.load("en_core_web_lg", disable=["parser", "ner"]) # disable=["parser", "tagger", "ner"]
except Exception:
    nltk.download("stopwords")
    nltk.download("punkt")
    spacy.cli.download("en_core_web_lg")
    tokenizer = nltk.tokenize.RegexpTokenizer(r"\w+'?\w+|\w+'")
    tokenizer.tokenize("I'm studying NLP and it's interesting.")
    stop_words = nltk.corpus.stopwords.words("english")
    nlp = spacy.load("en_core_web_lg", disable=["parser", "ner"])  # disable=["parser", "tagger", "ner"]
    print(stop_words[:20])
    print(len(stop_words))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']
198


In [6]:
# Create our corpus for training and perform some classic NLP preprocessing
dataset = pd.read_csv("/content/twitter.csv")

In [55]:
text_data = list(
    map(lambda x: tokenizer.tokenize(x.lower()), dataset['text'])
)
text_data = [
    [token.lemma_ for word in text for token in nlp(word)] for text in text_data
]
label_data = list(map(lambda x: x, dataset["feeling"]))
assert len(text_data) == len(label_data), f"{len(text_data)} does not equal {len(label_data)}"

In [56]:
EMBEDDING_DIM = 100
model = Word2Vec(
    text_data, vector_size=EMBEDDING_DIM, window=5, min_count=1, workers=4
)
word_vectors = model.wv
print(f"Vocabulary Length: {len(model.wv)}")
del model

Vocabulary Length: 626


In [57]:
padding_value = len(word_vectors.index_to_key)
# Embeddings are needed to give semantic value to the inputs of an LSTM
embedding_weights = torch.Tensor(word_vectors.vectors)

In [76]:
# RNN model

class RNN(nn.Module):

    def __init__(self,
                 input_dim,
                 embedding_dim,
                 hidden_dim,
                 output_dim,
                 n_layers,
                 bidirectional,
                 dropout,
                 embedding_weights):

        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            embedding_weights
        )

        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            dropout=dropout
        )

        self.fc = nn.Linear(
            hidden_dim * 2 if bidirectional else hidden_dim,
            output_dim
        )

        # 추가 필요
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, text_lengths):

        embedded = self.embedding(x)

        packed_embedded = nn.utils.rnn.pack_padded_sequence(
            embedded,
            text_lengths.cpu(),
            enforce_sorted=False
        )

        packed_output, hidden = self.rnn(packed_embedded)

        output, output_lengths = nn.utils.rnn.pad_packed_sequence(
            packed_output
        )

        if self.rnn.bidirectional:
            hidden = self.dropout(
                torch.cat((hidden[-2], hidden[-1]), dim=1)
            )
        else:
            hidden = self.dropout(hidden[-1])

        return self.fc(hidden)

# class RNN(nn.Module):
#     def __init__(
#             self,
#             input_dim,
#             embedding_dim,
#             hidden_dim,
#             output_dim,
#             embedding_weights
#     ):
#         super().__init__()
#         self.embedding = nn.Embedding.from_pretrained(embedding_weights)
#         self.rnn = nn.RNN(embedding_dim, hidden_dim)
#         self.fc = nn.Linear(hidden_dim, output_dim)

#     def forward(self, x, text_lengths):

#         embedded = self.embedding(x)

#         packed_embedded = nn.utils.rnn.pack_padded_sequence(
#             embedded,
#             text_lengths.cpu(),
#             enforce_sorted=False
#         )

#         packed_output, hidden = self.rnn(packed_embedded)

#         output, output_lengths = nn.utils.rnn.pad_packed_sequence(
#             packed_output
#         )

#         hidden = self.dropout(hidden[-1])

#         return self.fc(hidden)

    # def forward(self, x, text_lengths):
    #     embedded = self.embedding(x)
    #     packed_embedded = nn.utils.rnn.pad_packed_sequence(embedded, text_lengths)
    #     packed_output, hidden = self.rnn(packed_embedded)
    #     output, output_lengths = nn.utils.rnn.pad_packed_sequence(packed_output)

    #     return self.fc(hidden.squeeze(0))

In [77]:
INPUT_DIM = padding_value
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
OUTPUT_DIM = 1

In [79]:
rnn_model = RNN(
    INPUT_DIM,
    EMBEDDING_DIM,
    HIDDEN_DIM,
    OUTPUT_DIM,
    N_LAYERS,
    BIDIRECTIONAL,
    DROPOUT,
    embedding_weights
)

In [80]:
rnn_optimizer = optim.SGD(rnn_model.parameters(), lr=1e-3)
rnn_criterion = nn.BCEWithLogitsLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [81]:
# LSTM model
class LSTM(nn.Module):
    def __init__(self,
                 input_dim,
                 embedding_dim,
                 hidden_dim,
                 output_dim,
                 n_layers,
                 bidirectional,
                 dropout,
                 embedding_weights):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_weights)
        self.rnn = nn.LSTM(embedding_dim,
                           hidden_dim,
                           num_layers=n_layers,
                           bidirectional=bidirectional,
                           dropout=dropout)
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, text_lengths):

        embedded = self.embedding(x)

        packed_embedded = nn.utils.rnn.pack_padded_sequence(
            embedded,
            text_lengths.cpu(),
            enforce_sorted=False
        )

        packed_output, (hidden, cell) = self.rnn(packed_embedded)

        output, output_lengths = nn.utils.rnn.pad_packed_sequence(
            packed_output
        )

        hidden = self.dropout(
            torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        )

        return self.fc(hidden)

    # def forward(self, x, text_lengths):
    #     embedded = self.embedding(x)
    #     packed_embedded = nn.utils.rnn.pack_padded_sequence(
    #         embedded, text_lengths
    #     )
    #     packed_output, (hidden, cell) = self.rnn(packed_embedded)
    #     hidden = self.dropout(
    #         torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
    #     )
    #     return self.fc(hidden.squeeze(0))

In [82]:
INPUT_DIM = padding_value
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
OUTPUT_DIM = 1
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.5

In [83]:
lstm_model = LSTM(INPUT_DIM,
                  EMBEDDING_DIM,
                  HIDDEN_DIM,
                  OUTPUT_DIM,
                  N_LAYERS,
                  BIDIRECTIONAL,
                  DROPOUT,
                  embedding_weights)

In [84]:
lstm_optimizer = optim.Adam(lstm_model.parameters())
lstm_criterion = nn.BCEWithLogitsLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [85]:
def binary_accuracy(preds, y):
    rounded_preds = torch.round(torch.sigmoid(preds))
    correct = (rounded_preds == y).float()
    acc = correct.sum() / len(correct)
    return acc

In [86]:
def train(model, iterator, optimizer, criterion):
    epoch_loss = 0
    epoch_acc = 0
    model.train()
    for batch in iterator:
        optimizer.zero_grad()
        predictions = model(batch["text"], batch["length"]).squeeze(1)
        loss = criterion(predictions, batch["label"])
        acc = binary_accuracy(predictions, batch["label"])
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

In [87]:
def evaluate(model, iterator, criterion):
    epoch_loss = 0
    epoch_acc = 0
    model.eval()
    with torch.no_grad():
        for batch in iterator:
            predictions = model(batch["text"], batch["length"]).squeeze(1)
            loss = criterion(predictions, batch["label"])
            acc = binary_accuracy(predictions, batch["label"])

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

In [88]:
# Usually should be a power of 2 because it's the easiest for computer memory.
batch_size = 2

In [89]:
def iterator(X, y):
    size = len(X)
    permutation = np.random.permutation(size)
    iterate = []
    for i in range(0, size, batch_size):
        indices = permutation[i : i + batch_size]
        batch = {}
        batch["text"] = [X[i] for i in indices]
        batch["label"] = [y[i] for i in indices]

        batch["text"], batch["label"] = zip(
            *sorted(
                zip(batch["text"], batch["label"]),
                key=lambda x: len(x[0]),
                reverse=True,
            )
        )
        batch["length"] = [len(utt) for utt in batch["text"]]
        batch["length"] = torch.IntTensor(batch["length"])
        batch["text"] = torch.nn.utils.rnn.pad_sequence(
            batch["text"], batch_first=True
        ).t()
        batch["label"] = torch.Tensor(batch["label"])

        batch["label"] = batch["label"].to(device)
        batch["length"] = batch["length"].to(device)
        batch["text"] = batch["text"].to(device)

        iterate.append(batch)

    return iterate

In [90]:
index_utt = [
    torch.tensor([word_vectors.key_to_index.get(word, 0) for word in text])
    for text in text_data
]

In [91]:
# You've got to determine some labels for whatever you're training on.
X_train, X_test, y_train, y_test = train_test_split(
    index_utt, label_data, test_size=0.2
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2
)

In [92]:
train_iterator = iterator(X_train, y_train)
validate_iterator = iterator(X_val, y_val)
test_iterator = iterator(X_test, y_test)

print(len(train_iterator), len(validate_iterator), len(test_iterator))

32 8 10


In [93]:
N_EPOCHS = 25

In [94]:
for model in [rnn_model, lstm_model]:
    print(
        "|-----------------------------------------------------------------------------------------|"
    )
    print(f"Training with {model.__class__.__name__}")
    if "RNN" in model.__class__.__name__:
        for epoch in range(N_EPOCHS):
            train_loss, train_acc = train(
                rnn_model, train_iterator, rnn_optimizer, rnn_criterion
            )
            valid_loss, valid_acc = evaluate(
                rnn_model, validate_iterator, rnn_criterion
            )

            print(
                f"| Epoch: {epoch+1:02} | Train Loss: {train_loss: .3f} | Train Acc: {train_acc*100: .2f}% | Validation Loss: {valid_loss: .3f} | Validation Acc: {valid_acc*100: .2f}% |"
            )
    else:
        for epoch in range(N_EPOCHS):
            train_loss, train_acc = train(
                lstm_model, train_iterator, lstm_optimizer, lstm_criterion
            )
            valid_loss, valid_acc = evaluate(
                lstm_model, validate_iterator, lstm_criterion
            )

            print(
                f"| Epoch: {epoch+1:02} | Train Loss: {train_loss: .3f} | Train Acc: {train_acc*100: .2f}% | Validation Loss: {valid_loss: .3f} | Validation Acc: {valid_acc*100: .2f}% |"
            )

|-----------------------------------------------------------------------------------------|
Training with RNN
| Epoch: 01 | Train Loss:  0.695 | Train Acc:  48.44% | Validation Loss:  0.693 | Validation Acc:  50.00% |
| Epoch: 02 | Train Loss:  0.697 | Train Acc:  45.31% | Validation Loss:  0.693 | Validation Acc:  56.25% |
| Epoch: 03 | Train Loss:  0.696 | Train Acc:  46.88% | Validation Loss:  0.692 | Validation Acc:  56.25% |
| Epoch: 04 | Train Loss:  0.703 | Train Acc:  42.19% | Validation Loss:  0.692 | Validation Acc:  56.25% |
| Epoch: 05 | Train Loss:  0.696 | Train Acc:  51.56% | Validation Loss:  0.691 | Validation Acc:  56.25% |
| Epoch: 06 | Train Loss:  0.702 | Train Acc:  40.62% | Validation Loss:  0.691 | Validation Acc:  56.25% |
| Epoch: 07 | Train Loss:  0.688 | Train Acc:  54.69% | Validation Loss:  0.690 | Validation Acc:  56.25% |
| Epoch: 08 | Train Loss:  0.692 | Train Acc:  59.38% | Validation Loss:  0.690 | Validation Acc:  56.25% |
| Epoch: 09 | Train Loss: 